In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.dataset import get_transforms, make_loaders
from src.evaluate import get_logits, predict_loader, summarize
from src.experiment import run_experiment, run_experiment_tl
from src.model import CNNBaseline, ResNetFromScratch, TLResNet, describe_run
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()
paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [ ]:
img_size = 224
_, eval_tf = get_transforms(img_size, augment=False)
train_ds = datasets.ImageFolder('../data/processed/train', transform=eval_tf)
val_ds   = datasets.ImageFolder('../data/processed/val', transform=eval_tf)
g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

## feature extraction :

In [ ]:
set_seed(SEED)

model = TLResNet(trainable=('fc',)).to(device)
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

res_fe = run_experiment_tl(
    name='resnet_fe',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr_head': 1e-3, 'trainable': 'fc', 'img_size': 224},
)

## fine-tuning:

In [ ]:
set_seed(SEED)

model = TLResNet(trainable=('fc',)).to(device)
ckpt = torch.load('../checkpoints/resnet_fe.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])   # ← اینجا
model.set_trainable(('layer4', 'fc'))
opt = torch.optim.Adam([
    {'params': model.net.layer4.parameters(), 'lr': 1e-4},
    {'params': model.net.fc.parameters(),     'lr': 1e-3},
])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,}")

res_ft = run_experiment_tl(
    name='resnet_ft',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr_layer4': 1e-4, 'lr_head': 1e-3, 'trainable': 'layer4+fc', 'img_size': 224},
)

print(res_ft)

## Stretch: ResNet:

In [ ]:
tf_small = transforms.Compose([
    transforms.Resize((32, 32)),   
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_ds_small = datasets.ImageFolder('../data/processed/train', transform=tf_small)
val_ds_small = datasets.ImageFolder('../data/processed/val', transform=tf_small)

train_loader_small = DataLoader(train_ds_small, batch_size=64, shuffle=True)
val_loader_small = DataLoader(val_ds_small, batch_size=64, shuffle=False)

In [21]:
set_seed(SEED)

model = ResNetFromScratch(num_classes=8).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable:,}")

res_scratch = run_experiment_tl(
    name='resnet_scratch',
    model=model,
    train_loader=train_loader_small,
    val_loader=val_loader_small,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr': 1e-3, 'trainable': 'all', 'img_size': 32, 'pretrained': False},
)

print(res_scratch)

trainable: 11,172,936
{'epoch': 1, 'train_loss': 1.5960220862168966, 'val_loss': 2.3125320292533713, 'val_macro_f1': 0.256168651533372, 'lr': 0.001}
{'epoch': 2, 'train_loss': 0.8889773869528581, 'val_loss': 1.4587233503377748, 'val_macro_f1': 0.4686645022950038, 'lr': 0.001}
{'epoch': 3, 'train_loss': 0.652546789746372, 'val_loss': 1.2153999007903655, 'val_macro_f1': 0.589321907940329, 'lr': 0.001}
{'epoch': 4, 'train_loss': 0.5133755312931474, 'val_loss': 0.9850930081191638, 'val_macro_f1': 0.66112717311867, 'lr': 0.001}
{'epoch': 5, 'train_loss': 0.32356446857915394, 'val_loss': 0.8422546121809218, 'val_macro_f1': 0.7087507605836967, 'lr': 0.001}
{'epoch': 6, 'train_loss': 0.2746554634287345, 'val_loss': 0.6477641853598556, 'val_macro_f1': 0.7634548585739224, 'lr': 0.001}
{'epoch': 7, 'train_loss': 0.20450995429300428, 'val_loss': 0.8168329919201827, 'val_macro_f1': 0.7382581560941894, 'lr': 0.001}
{'epoch': 8, 'train_loss': 0.14575999282640137, 'val_loss': 0.7682581319876597, 'val_